# Learnable-R + IMU-only CDAN — 아키텍처 다이어그램

학습 가능한 회전 R 과 IMU 단독 CDAN 모델의 구조를 그림(PNG)으로 출력한다.
- **Cell 2**: matplotlib 스키매틱 (의존성 없음) → `results/Learnable_R/architecture.png`
- **Cell 3**: (선택) 실제 `nn.Module` 그래프를 torchview 로 덤프 (설치되어 있을 때만)

흐름: `target IMU → R(SO3) → post-R BN → IMU Encoder(TCN) → Fusion TCN → feature(512) → {label head, GRL+CDAN domain head}`.
source 는 R 을 우회(apply_r=False).

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

# 한글 폰트: koreanize_matplotlib 가 있으면 그걸로(가장 간단·portable),
# 없으면 Mac 시스템 폰트로 fallback.
try:
    import koreanize_matplotlib  # noqa: F401  (import 만으로 한글 폰트 자동 설정)
    print('font: koreanize_matplotlib')
except ImportError:
    from matplotlib import font_manager as fm
    _avail = {f.name for f in fm.fontManager.ttflist}
    for _f in ['Apple SD Gothic Neo', 'AppleGothic', 'Nanum Gothic', 'Noto Sans CJK KR']:
        if _f in _avail:
            matplotlib.rcParams['font.family'] = _f
            print('font:', _f)
            break
    else:
        print('한글 폰트 미발견 — 글자가 깨질 수 있음')
matplotlib.rcParams['axes.unicode_minus'] = False  # NanumGothic 에 U+2212 없음 → ASCII '-' 사용

OUT = os.path.join('..', 'results', 'Learnable_R', 'architecture.png')
os.makedirs(os.path.dirname(OUT), exist_ok=True)

C = {
    'in':   '#E8EEF7',
    'R':    '#FFE0B2',   # learnable R 강조
    'norm': '#E6F4EA',
    'enc':  '#D6E4F0',
    'fus':  '#C9DBEF',
    'head': '#EADCF6',
    'grl':  '#FAD4D4',
    'out':  '#F5F5F5',
    'loss': '#FFF7DA',
}

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16); ax.set_ylim(0, 9); ax.axis('off')

def box(x, y, w, h, text, color, fs=10, bold=False):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.04,rounding_size=0.12',
                       linewidth=1.4, edgecolor='#3b3b3b', facecolor=color)
    ax.add_patch(p)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center', fontsize=fs,
            fontweight='bold' if bold else 'normal')
    return (x, y, w, h)

def arrow(p1, p2, text='', color='#2b2b2b', rad=0.0, side='top'):
    a = FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=16,
                        linewidth=1.6, color=color,
                        connectionstyle=f'arc3,rad={rad}')
    ax.add_patch(a)
    if text:
        mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
        ax.text(mx, my + (0.28 if side=='top' else -0.32), text, ha='center',
                va='center', fontsize=8.5, color=color)

def rc(b):   # right-center
    return (b[0]+b[2], b[1]+b[3]/2)
def lc(b):   # left-center
    return (b[0], b[1]+b[3]/2)
def tc(b):
    return (b[0]+b[2]/2, b[1]+b[3])
def bc(b):
    return (b[0]+b[2]/2, b[1])

y0 = 5.2; h = 1.25
b_in  = box(0.2, y0, 2.0, h, 'target IMU\n(B,3,500)\n100Hz x 5s', C['in'], 9)
b_R   = box(2.7, y0, 2.0, h, 'Learnable R\nSO(3): R=exp(w^)\nw in R^3, init=I', C['R'], 9, bold=True)
b_bn  = box(5.2, y0, 1.9, h, 'post-R\nBatchNorm1d(3)\n(회전→정규화)', C['norm'], 8.5)
b_enc = box(7.6, y0, 2.5, h, 'IMU Encoder (TCN)\nstem Conv(3->64,k11,s5)\n+4 Residual-TCN(SE)\n-> (B,256,50)', C['enc'], 8)
b_fus = box(10.6, y0, 2.6, h, 'Fusion TCN\nConv(256->64) +6 Res-TCN\n(dil 1,2,4,8,16,32)\n+AvgPool -> (B,512)', C['fus'], 8)
b_feat= box(13.7, y0, 2.1, h, 'feature\n(B,512)', C['out'], 9, bold=True)

for a, b in [(b_in,b_R),(b_R,b_bn),(b_bn,b_enc),(b_enc,b_fus),(b_fus,b_feat)]:
    arrow(rc(a), lc(b))

# source bypass (R 우회)
ax.text(3.7, y0+h+0.55, 'source: apply_r=False (R 우회)   |   target: apply_r=True',
        ha='center', fontsize=9, style='italic', color='#555')
arrow((b_in[0]+b_in[2], y0+h-0.2), (b_bn[0]+0.2, y0+h-0.2), color='#888', rad=0.35)

# heads
b_lab = box(13.4, 7.2, 2.4, 1.0, 'Label head\nLinear 512->256->10', C['head'], 8.5)
ax.text(14.9, 8.5, 'class logits (10)', ha='center', fontsize=8.5, fontweight='bold')

b_grl = box(11.2, 2.0, 1.7, 1.0, 'GRL\n(reverse, alpha)', C['grl'], 8.5)
b_dom = box(13.2, 2.0, 2.6, 1.0, 'CDAN Domain head\nf (x) softmax(c) ->5120\n->1024->512->2', C['head'], 7.8)
ax.text(14.5, 1.55, 'domain logits (2)', ha='center', fontsize=8.5, fontweight='bold')

# feature -> heads
arrow(tc(b_feat), (b_lab[0]+b_lab[2]/2, b_lab[1]), 'classify', rad=-0.15)
arrow(bc(b_feat), (b_grl[0]+b_grl[2], b_grl[1]+b_grl[3]/2), 'adversarial', rad=0.2, side='bot')
arrow(rc(b_grl), lc(b_dom))
arrow(tc(b_lab), (14.4, 8.45), '', rad=0)
arrow(bc(b_dom), (14.5, 1.7), '', rad=0)

# loss panel (NanumGothic 에 없는 글자(−, ᵀ)는 ASCII '-', '^T' 로 표기)
lx, ly, lw, lh = 0.2, 0.3, 9.8, 2.7
box(lx, ly, lw, lh, '', C['loss'], 9)
ax.text(lx+0.2, ly+lh-0.3, 'R 을 움직이는 손실 (매 배치 · 주입 아님 · R=I 에서 출발)',
        ha='left', va='center', fontsize=10, fontweight='bold')
lines = [
    'L_total = L_cls (source CE)',
    '   + lam_da  · L_domain  (CDAN/DANN 적대, GRL)         -> 인코더 target 적응(정확도 레버)',
    '   + lam_pca · ‖R F_t - F_s‖²  (on-the-fly PCA 주축)    -> yaw 관측, R 을 perm 으로 이동',
    '   + lam_g   · ‖R g_t - g_s‖²  (gravity, pitch/roll)    -> yaw盲, PCA 와 충돌 가능',
    '   + lam_rot · (‖R^T R - I‖² + |det R - 1|²)            -> SO(3) 에서 구조적 0',
    '   + lam_cons· L_consistency  (원본 vs 회전증강 target 예측 KL)',
]
for i, t in enumerate(lines):
    ax.text(lx+0.35, ly+lh-0.78-i*0.38, t, ha='left', va='center', fontsize=9)

ax.set_title('Learnable-R + IMU-only CDAN  (target IMU 정렬용 학습 회전 R)',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(OUT, dpi=150, bbox_inches='tight')
print('saved ->', os.path.abspath(OUT))
plt.show()

## (선택) 실제 nn.Module 그래프 덤프

`torchview` 가 설치되어 있으면 진짜 모델 연산 그래프를 그린다 (`pip install torchview` + graphviz).
설치 안 돼 있으면 조용히 건너뛴다.

In [ ]:
import os, sys
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath(os.path.join('..', 'Multimodal')))
try:
    import torch
    from learnable_r_model import LearnableRCDAN
    m = LearnableRCDAN(num_classes=10, post_r_norm='batchnorm').eval()
    emg = torch.randn(2, 2, 5000)
    imu = torch.randn(2, 3, 500)
    try:
        from torchview import draw_graph
        g = draw_graph(m, input_data=(emg, imu, 0.5, True), expand_nested=True,
                       graph_name='LearnableRCDAN', save_graph=True,
                       directory=os.path.join('..', 'results', 'Learnable_R'),
                       filename='architecture_torchview')
        print('torchview saved -> results/Learnable_R/architecture_torchview.png')
    except ImportError:
        print('torchview 미설치 — 건너뜀. (pip install torchview, 그리고 graphviz 필요)')
    # 어쨌든 shape 검증 출력
    c, d, f = m(emg, imu, alpha=0.5, apply_r=True)
    print('class', tuple(c.shape), '| domain', tuple(d.shape), '| feature', tuple(f.shape))
    n = sum(p.numel() for p in m.parameters())
    print(f'total params: {n:,}  |  R params: {m.r.w.numel()} (SO(3) 3-vector)')
except Exception as e:
    print('skip (모델 import 실패 또는 torch 없음):', repr(e))